In [1]:
import pandas as pd

sales = pd.read_csv("bm_sales.csv")

print("Shape:", sales.shape)
sales.head()

Shape: (641843, 9)


,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct
0,2021-01-01,26,1124,2961.0,1,20.08,20.08,Store,15.0
1,2021-01-01,19,1035,NaN,3,208.57,625.71,Store,15.0
2,2021-01-01,38,1088,NaN,1,17.99,17.99,Website,15.0
3,2021-01-01,3,1164,3694.0,6,196.24,1177.44,Website,0.0
4,2021-01-01,2,1093,1252.0,1,7.98,7.98,Store,0.0


In [2]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641843 entries, 0 to 641842
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   date          641843 non-null  object 
 1   store_id      641843 non-null  int64  
 2   sku_id        641843 non-null  int64  
 3   customer_id   482022 non-null  float64
 4   quantity      641843 non-null  int64  
 5   unit_price    641843 non-null  float64
 6   total_value   641843 non-null  float64
 7   channel       641843 non-null  object 
 8   discount_pct  641843 non-null  float64
dtypes: float64(4), int64(3), object(2)
memory usage: 44.1+ MB


In [3]:
sales.duplicated().sum()

np.int64(45)

In [4]:
sales[sales.duplicated()]

,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct
5783,2021-01-12,28,1061,NaN,2,45.75,91.50,Store,15.0
31306,2021-03-06,10,1037,NaN,1,178.57,178.57,Store,0.0
66573,2021-06-06,10,1145,NaN,1,11.46,11.46,Store,0.0
71332,2021-06-19,13,1024,NaN,2,8.79,17.58,Store,0.0
87384,2021-07-31,27,1069,NaN,5,34.47,172.35,MobileApp,10.0
106520,2021-10-01,33,1045,NaN,1,141.21,141.21,Website,0.0
107188,2021-10-04,2,1159,NaN,3,178.08,534.24,Store,0.0
126550,2021-12-11,46,1146,NaN,1,20.29,20.29,Website,0.0
140138,2022-01-13,35,1133,NaN,3,25.39,76.17,Store,25.0
147735,2022-01-29,25,1107,NaN,3,120.58,361.74,Store,25.0


In [5]:
sales.isnull().sum()

date                 0
store_id             0
sku_id               0
customer_id     159821
quantity             0
unit_price           0
total_value          0
channel              0
discount_pct         0
dtype: int64

In [6]:
sales["calculated_value"] = (
    sales["quantity"] * sales["unit_price"]
)

In [7]:
sales["difference"] = (
    sales["total_value"] - sales["calculated_value"]
)

sales["difference"].describe()

count    6.418430e+05
mean    -2.781550e-16
std      1.228716e-14
min     -2.273737e-13
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.273737e-13
Name: difference, dtype: float64

In [8]:
(sales["difference"].abs() > 0.01).sum()

np.int64(0)

In [9]:
print("Quantity <= 0:", (sales["quantity"] <= 0).sum())
print("Price <= 0:", (sales["unit_price"] <= 0).sum())
print("Discount < 0:", (sales["discount_pct"] < 0).sum())
print("Discount > 100:", (sales["discount_pct"] > 100).sum())

Quantity <= 0: 0
Price <= 0: 0
Discount < 0: 0
Discount > 100: 0


In [10]:
sales["date"] = pd.to_datetime(sales["date"])

print("Minimum date:", sales["date"].min())
print("Maximum date:", sales["date"].max())

Minimum date: 2021-01-01 00:00:00
Maximum date: 2025-10-31 00:00:00


In [11]:
duplicates = sales[sales.duplicated(keep=False)]

duplicates.sort_values(
    by=["date", "store_id", "sku_id"]
).head(20)

,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,calculated_value,difference
5463,2021-01-12,28,1061,NaN,2,45.75,91.50,Store,15.0,91.50,0.0
5783,2021-01-12,28,1061,NaN,2,45.75,91.50,Store,15.0,91.50,0.0
31125,2021-03-06,10,1037,NaN,1,178.57,178.57,Store,0.0,178.57,0.0
31306,2021-03-06,10,1037,NaN,1,178.57,178.57,Store,0.0,178.57,0.0
66354,2021-06-06,10,1145,NaN,1,11.46,11.46,Store,0.0,11.46,0.0
66573,2021-06-06,10,1145,NaN,1,11.46,11.46,Store,0.0,11.46,0.0
71206,2021-06-19,13,1024,NaN,2,8.79,17.58,Store,0.0,17.58,0.0
71332,2021-06-19,13,1024,NaN,2,8.79,17.58,Store,0.0,17.58,0.0
87236,2021-07-31,27,1069,NaN,5,34.47,172.35,MobileApp,10.0,172.35,0.0
87384,2021-07-31,27,1069,NaN,5,34.47,172.35,MobileApp,10.0,172.35,0.0


In [12]:
duplicates.shape

(89, 11)

In [13]:
duplicates["customer_id"].isna().sum()

np.int64(87)

In [14]:
duplicates.groupby(
    ["date", "store_id", "sku_id", "customer_id",
     "quantity", "unit_price", "total_value",
     "channel", "discount_pct"]
).size().value_counts()

2    1
Name: count, dtype: int64

In [15]:
duplicates.groupby(
    ["date", "store_id", "sku_id", "customer_id",
     "quantity", "unit_price", "total_value",
     "channel", "discount_pct"],
    dropna=False
).size().value_counts()

2    43
3     1
Name: count, dtype: int64

In [16]:
duplicates.groupby(
    ["date", "store_id", "sku_id", "customer_id",
     "quantity", "unit_price", "total_value",
     "channel", "discount_pct"],
    dropna=False
).size().sort_values(ascending=False).head(10)

date        store_id  sku_id  customer_id  quantity  unit_price  total_value  channel    discount_pct
2025-05-14  31        1176    NaN          3         6.83        20.49        MobileApp  0.0             3
2021-01-12  28        1061    NaN          2         45.75       91.50        Store      15.0            2
2021-06-06  10        1145    NaN          1         11.46       11.46        Store      0.0             2
2021-03-06  10        1037    NaN          1         178.57      178.57       Store      0.0             2
2021-07-31  27        1069    NaN          5         34.47       172.35       MobileApp  10.0            2
2021-10-01  33        1045    NaN          1         141.21      141.21       Website    0.0             2
2021-10-04  2         1159    NaN          3         178.08      534.24       Store      0.0             2
2021-12-11  46        1146    NaN          1         20.29       20.29        Website    0.0             2
2022-01-13  35        1133    NaN         

In [17]:
sales = sales.drop_duplicates()

In [18]:
print("New shape:", sales.shape)
print("Remaining duplicates:", sales.duplicated().sum())

New shape: (641798, 11)
Remaining duplicates: 0


In [19]:
import pandas as pd

skus = pd.read_csv("bm_skus.csv")
stores = pd.read_csv("bm_stores.csv")
customers = pd.read_csv("bm_customers.csv")
inventory = pd.read_csv("bm_inventory.csv")
promotions = pd.read_csv("bm_promotions.csv")

In [20]:
# Check SKU relationships
print("Sales SKUs not in SKU master:",
      (~sales["sku_id"].isin(skus["sku_id"])).sum())

# Check Store relationships
print("Sales stores not in Store master:",
      (~sales["store_id"].isin(stores["store_id"])).sum())

# Check Customer relationships
customer_ids = customers["cust_id"]

print("Sales customers not in Customer master:",
      (
          sales["customer_id"].notna() &
          ~sales["customer_id"].isin(customer_ids)
      ).sum())

# Check Inventory SKUs
print("Inventory SKUs not in SKU master:",
      (~inventory["sku_id"].isin(skus["sku_id"])).sum())

# Check Inventory stores
print("Inventory stores not in Store master:",
      (~inventory["store_id"].isin(stores["store_id"])).sum())

Sales SKUs not in SKU master: 0
Sales stores not in Store master: 0
Sales customers not in Customer master: 0
Inventory SKUs not in SKU master: 0
Inventory stores not in Store master: 0


In [21]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8735 entries, 0 to 8734
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   store_id           8735 non-null   int64 
 1   sku_id             8735 non-null   int64 
 2   stock_on_hand      8735 non-null   int64 
 3   reorder_point      8735 non-null   int64 
 4   safety_stock       8735 non-null   int64 
 5   last_restock_date  8735 non-null   object
 6   snapshot_date      8735 non-null   object
dtypes: int64(5), object(2)
memory usage: 477.8+ KB


In [22]:
inventory.isnull().sum()

store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
snapshot_date        0
dtype: int64

In [23]:
inventory.describe()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock
count,8735.000000,8735.000000,8735.000000,8735.000000,8735.000000
mean,25.343102,1100.295478,170.763938,67.755238,33.629193
std,14.385357,57.629837,77.875511,32.988546,16.499030
min,1.000000,1001.000000,35.000000,10.000000,5.000000
25%,13.000000,1051.000000,112.000000,43.000000,21.000000
50%,25.000000,1100.000000,156.000000,62.000000,31.000000
75%,38.000000,1150.000000,217.000000,87.000000,43.000000
max,50.000000,1200.000000,449.000000,217.000000,108.000000


In [24]:
inventory.duplicated().sum()

np.int64(0)

In [25]:
print(
    "Items below reorder point:",
    (inventory["stock_on_hand"] < inventory["reorder_point"]).sum()
)

Items below reorder point: 0


In [26]:
print(
    "Percentage below reorder point:",
    round(
        (inventory["stock_on_hand"] < inventory["reorder_point"]).mean() * 100,
        2
    ),
    "%"
)

Percentage below reorder point: 0.0 %


In [27]:
print("Negative stock:", (inventory["stock_on_hand"] < 0).sum())
print("Negative reorder point:", (inventory["reorder_point"] < 0).sum())
print("Negative safety stock:", (inventory["safety_stock"] < 0).sum())

Negative stock: 0
Negative reorder point: 0
Negative safety stock: 0


In [28]:
inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"]
)

inventory["snapshot_date"] = pd.to_datetime(
    inventory["snapshot_date"]
)

In [29]:
print("Last restock:",
      inventory["last_restock_date"].min(),
      "to",
      inventory["last_restock_date"].max())

print("Snapshot:",
      inventory["snapshot_date"].min(),
      "to",
      inventory["snapshot_date"].max())

Last restock: 2025-08-03 00:00:00 to 2025-10-30 00:00:00
Snapshot: 2025-10-31 00:00:00 to 2025-10-31 00:00:00


In [30]:
print(
    "Safety stock > Reorder point:",
    (inventory["safety_stock"] > inventory["reorder_point"]).sum()
)

print(
    "Stock below safety stock:",
    (inventory["stock_on_hand"] < inventory["safety_stock"]).sum()
)

Safety stock > Reorder point: 0
Stock below safety stock: 0


In [31]:
print("SKU table")
print("Duplicates:", skus.duplicated().sum())
print("Missing values:")
print(skus.isnull().sum())

print("\nStore table")
print("Duplicates:", stores.duplicated().sum())
print("Missing values:")
print(stores.isnull().sum())

print("\nCustomer table")
print("Duplicates:", customers.duplicated().sum())
print("Missing values:")
print(customers.isnull().sum())

SKU table
Duplicates: 0
Missing values:
sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64

Store table
Duplicates: 0
Missing values:
store_id        0
store_name      0
city            0
store_type      0
opening_date    0
dtype: int64

Customer table
Duplicates: 0
Missing values:
cust_id              0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64


In [32]:
print("Invalid SKU prices:")
print("Unit price <= 0:", (skus["unit_price"] <= 0).sum())
print("Cost price <= 0:", (skus["cost_price"] <= 0).sum())

print("\nCustomer age:")
print("Age <= 0:", (customers["age"] <= 0).sum())
print("Age > 100:", (customers["age"] > 100).sum())

Invalid SKU prices:
Unit price <= 0: 0
Cost price <= 0: 0

Customer age:
Age <= 0: 0
Age > 100: 0


In [33]:
stores["opening_date"] = pd.to_datetime(stores["opening_date"])
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"]
)

print("Store opening:", stores["opening_date"].min(),
      "to", stores["opening_date"].max())

print("Customer registration:", customers["registration_date"].min(),
      "to", customers["registration_date"].max())

Store opening: 2017-01-01 00:00:00 to 2018-01-01 00:00:00
Customer registration: 2021-01-01 00:00:00 to 2025-10-30 00:00:00


In [34]:
print("Shape:", promotions.shape)

print("\nDuplicates:")
print(promotions.duplicated().sum())

print("\nMissing values:")
print(promotions.isnull().sum())

print("\nDiscount validation:")
print("Discount < 0:", (promotions["discount_pct"] < 0).sum())
print("Discount > 100:", (promotions["discount_pct"] > 100).sum())

Shape: (33, 6)

Duplicates:
0

Missing values:
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
promo_id        0
dtype: int64

Discount validation:
Discount < 0: 0
Discount > 100: 0


In [35]:
print("Earliest promotion:", promotions["start_date"].min())
print("Latest promotion:", promotions["end_date"].max())

Earliest promotion: 2021-01-01
Latest promotion: 2025-08-31


In [36]:
print(
    "Invalid promotion periods:",
    (promotions["end_date"] < promotions["start_date"]).sum()
)

Invalid promotion periods: 0


In [37]:
print(
    "Promotions with discount > 0:",
    (promotions["discount_pct"] > 0).sum()
)

Promotions with discount > 0: 33


In [38]:
# Remove temporary columns we created during validation
sales = sales.drop(columns=["calculated_value", "difference"])

# Save cleaned files
sales.to_csv("cleaned_sales.csv", index=False)
skus.to_csv("cleaned_skus.csv", index=False)
stores.to_csv("cleaned_stores.csv", index=False)
customers.to_csv("cleaned_customers.csv", index=False)
inventory.to_csv("cleaned_inventory.csv", index=False)
promotions.to_csv("cleaned_promotions.csv", index=False)

In [39]:
import os

for file in [
    "cleaned_sales.csv",
    "cleaned_skus.csv",
    "cleaned_stores.csv",
    "cleaned_customers.csv",
    "cleaned_inventory.csv",
    "cleaned_promotions.csv"
]:
    print(file, "->", os.path.exists(file))

cleaned_sales.csv -> True
cleaned_skus.csv -> True
cleaned_stores.csv -> True
cleaned_customers.csv -> True
cleaned_inventory.csv -> True
cleaned_promotions.csv -> True


In [40]:
import os

os.listdir()

['.ipynb_checkpoints',
 'bm_customers.csv',
 'bm_inventory.csv',
 'bm_promotions.csv',
 'bm_sales.csv',
 'bm_skus.csv',
 'bm_stores.csv',
 'cleaned_customers.csv',
 'cleaned_inventory.csv',
 'cleaned_promotions.csv',
 'cleaned_sales.csv',
 'cleaned_skus.csv',
 'cleaned_stores.csv',
 'Untitled.ipynb']

In [45]:
import os

print(os.getcwd())
print(os.path.abspath("cleaned_sales.csv"))

C:\Users\User\SQL
C:\Users\User\SQL\cleaned_sales.csv


In [46]:
sales["customer_id"] = sales["customer_id"].astype("Int64")

In [47]:
sales.to_csv("cleaned_sales.csv", index=False, na_rep="")

In [48]:
import os

print(os.getcwd())
print(os.path.abspath("cleaned_sales.csv"))

C:\Users\User\SQL
C:\Users\User\SQL\cleaned_sales.csv


In [49]:
# Convert customer IDs to proper integers while keeping missing values
sales["customer_id"] = pd.to_numeric(
    sales["customer_id"],
    errors="coerce"
).astype("Int64")

# Save a fresh CSV specifically for MySQL
sales.to_csv(
    "cleaned_sales_mysql.csv",
    index=False,
    na_rep=""
)

In [50]:
print(sales[["customer_id"]].head(10))

   customer_id
0         2961
1         <NA>
2         <NA>
3         3694
4         1252
5         1286
6         <NA>
7         3160
8         3870
9         1033


In [51]:
pd.read_csv(
    "cleaned_sales_mysql.csv",
    dtype={"customer_id": "string"}
).head()

,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct
0,2021-01-01,26,1124,2961,1,20.08,20.08,Store,15.0
1,2021-01-01,19,1035,<NA>,3,208.57,625.71,Store,15.0
2,2021-01-01,38,1088,<NA>,1,17.99,17.99,Website,15.0
3,2021-01-01,3,1164,3694,6,196.24,1177.44,Website,0.0
4,2021-01-01,2,1093,1252,1,7.98,7.98,Store,0.0


In [52]:
check = pd.read_csv(
    "cleaned_sales_mysql.csv",
    dtype={"customer_id": "string"}
)

print(check.iloc[1364:1368][["date", "store_id", "sku_id", "customer_id"]])

            date  store_id  sku_id customer_id
1364  2021-01-03        25    1094        3807
1365  2021-01-03         7    1095        4109
1366  2021-01-03         4    1184        1353
1367  2021-01-03        34    1132        1774


In [53]:
print(repr(check.iloc[1365]["customer_id"]))

'4109'


In [54]:
bad = check[
    check["customer_id"].notna() &
    ~check["customer_id"].str.fullmatch(r"\d+")
]

print("Bad customer IDs:", len(bad))
print(bad[["date", "store_id", "sku_id", "customer_id"]].head(20))

Bad customer IDs: 0
Empty DataFrame
Columns: [date, store_id, sku_id, customer_id]
Index: []


In [55]:
print(check["customer_id"].value_counts(dropna=False).head(20))

customer_id
<NA>    159777
3385       129
3845       129
2652       128
3084       128
1535       128
1039       127
4955       127
3594       127
3544       127
1348       127
3680       126
1865       125
1585       125
301        125
4239       125
2578       125
1869       125
3151       124
3337       124
Name: count, dtype: Int64


In [56]:
import os

print(os.getcwd())
print(os.path.abspath("cleaned_sales_mysql.csv"))

C:\Users\User\SQL
C:\Users\User\SQL\cleaned_sales_mysql.csv


In [57]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/17.7 MB 493.7 kB/s eta 0:00:35
   - -------------------------------------- 0.5/17.7 MB 493.7 kB/s eta 0:00:35
   - -------------------------------------- 0.5/17.7 MB 493.7 kB/s eta 0:00:35
   - -------------------------------------- 0.8/17.7 MB 453.5 kB/s eta 0:00:38
   - -------------------------------------- 0.8/17.7 MB 453.5 kB/s eta 0:00:38
   -- ------------------------------------- 1.0/17.7 MB 484.0 kB/s eta 0:00:35
   -- ------------------------------------- 1.0/17.7 MB 484.0 kB/s eta 0:00:35
   -- ------------------------------------- 1.3/17.7 MB 512.5 kB/s eta 0:00:32
   -- ------------

In [63]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="project_user",
    password="likki78",
    database="retail_analytics"
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [64]:
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('retail_analytics',)


In [65]:
cursor.execute("TRUNCATE TABLE fact_sales")
conn.commit()

print("fact_sales cleared")

fact_sales cleared


In [66]:
import pandas as pd

file_path = r"C:\Users\User\SQL\cleaned_sales_mysql.csv"

insert_query = """
INSERT INTO fact_sales
(
    sale_date,
    store_id,
    sku_id,
    cust_id,
    quantity,
    unit_price,
    total_value,
    channel,
    discount_pct
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

total_inserted = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=5000,
    dtype={"customer_id": "Int64"}
):
    
    rows = []

    for _, row in chunk.iterrows():
        customer_id = (
            int(row["customer_id"])
            if pd.notna(row["customer_id"])
            else None
        )

        rows.append((
            row["date"],
            int(row["store_id"]),
            int(row["sku_id"]),
            customer_id,
            int(row["quantity"]),
            float(row["unit_price"]),
            float(row["total_value"]),
            row["channel"],
            float(row["discount_pct"])
        ))

    cursor.executemany(insert_query, rows)
    conn.commit()

    total_inserted += len(rows)

    print(f"Inserted: {total_inserted:,}")

Inserted: 5,000
Inserted: 10,000
Inserted: 15,000
Inserted: 20,000
Inserted: 25,000
Inserted: 30,000
Inserted: 35,000
Inserted: 40,000
Inserted: 45,000
Inserted: 50,000
Inserted: 55,000
Inserted: 60,000
Inserted: 65,000
Inserted: 70,000
Inserted: 75,000
Inserted: 80,000
Inserted: 85,000
Inserted: 90,000
Inserted: 95,000
Inserted: 100,000
Inserted: 105,000
Inserted: 110,000
Inserted: 115,000
Inserted: 120,000
Inserted: 125,000
Inserted: 130,000
Inserted: 135,000
Inserted: 140,000
Inserted: 145,000
Inserted: 150,000
Inserted: 155,000
Inserted: 160,000
Inserted: 165,000
Inserted: 170,000
Inserted: 175,000
Inserted: 180,000
Inserted: 185,000
Inserted: 190,000
Inserted: 195,000
Inserted: 200,000
Inserted: 205,000
Inserted: 210,000
Inserted: 215,000
Inserted: 220,000
Inserted: 225,000
Inserted: 230,000
Inserted: 235,000
Inserted: 240,000
Inserted: 245,000
Inserted: 250,000
Inserted: 255,000
Inserted: 260,000
Inserted: 265,000
Inserted: 270,000
Inserted: 275,000
Inserted: 280,000
Inserted: 28

In [67]:
cursor.execute("SELECT COUNT(*) FROM fact_sales")
print(cursor.fetchone()[0])

641798


In [68]:
cursor.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(cust_id) AS identified_customers,
    COUNT(*) - COUNT(cust_id) AS anonymous_sales
FROM fact_sales
""")

print(cursor.fetchone())

(641798, 482021, 159777)


In [69]:
cursor.execute("TRUNCATE TABLE fact_inventory")
conn.commit()

print("fact_inventory cleared")

fact_inventory cleared


In [70]:
inventory_file = r"C:\Users\User\SQL\cleaned_inventory.csv"

inventory_insert = """
INSERT INTO fact_inventory
(
    store_id,
    sku_id,
    stock_on_hand,
    reorder_point,
    safety_stock,
    last_restock_date,
    snapshot_date
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

inventory_rows = []

for _, row in pd.read_csv(inventory_file).iterrows():
    inventory_rows.append((
        int(row["store_id"]),
        int(row["sku_id"]),
        int(row["stock_on_hand"]),
        int(row["reorder_point"]),
        int(row["safety_stock"]),
        row["last_restock_date"],
        row["snapshot_date"]
    ))

cursor.executemany(inventory_insert, inventory_rows)
conn.commit()

print("Inventory rows inserted:", len(inventory_rows))

Inventory rows inserted: 8735


In [71]:
cursor.execute("SELECT COUNT(*) FROM fact_inventory")
print("Inventory rows:", cursor.fetchone()[0])

Inventory rows: 8735
